# Load and Process Collected Dataset

This notebook processes both Kaggle and collected datasets for training.

In [1]:
import os
from os.path import dirname

root_dir = dirname(os.getcwd())
os.chdir(root_dir)

In [2]:
import torch
import pandas as pd

from src.utils import *

## Load Collected Dataset

In [3]:
def load_collected_data_train(folder_path):
    """Load accelerometer data and keep only data before the first time gap."""
    accel_path = os.path.join(folder_path, 'Accelerometer.csv')
    df = pd.read_csv(accel_path)

    # Convert to kaggle format: time, ax, ay, az
    processed_df = pd.DataFrame({
        'time': df['timeIntervalSince1970'],
        'ax': df['x'],
        'ay': df['y'],
        'az': df['z']
    })
    processed_df = normalize_time(processed_df)
    
    # Detect time gaps > 1 second and keep only data before the first gap
    time_diff = processed_df['time'].diff()
    gap_indices = time_diff[time_diff > 1.0].index
    
    if len(gap_indices) > 0:
        first_gap_idx = gap_indices[0]
        print(f"  Train: Time gap detected at index {first_gap_idx}. Keeping first segment ({first_gap_idx} rows).")
        processed_df = processed_df.iloc[:first_gap_idx].reset_index(drop=True)
    
    return processed_df

def load_collected_data_test(folder_path):
    """Load accelerometer data and keep only data between first and second time gaps."""
    accel_path = os.path.join(folder_path, 'Accelerometer.csv')
    df = pd.read_csv(accel_path)

    # Convert to kaggle format: time, ax, ay, az
    processed_df = pd.DataFrame({
        'time': df['timeIntervalSince1970'],
        'ax': df['x'],
        'ay': df['y'],
        'az': df['z']
    })
    processed_df = normalize_time(processed_df)
    
    # Detect time gaps > 1 second
    time_diff = processed_df['time'].diff()
    gap_indices = time_diff[time_diff > 1.0].index
    
    if len(gap_indices) >= 2:
        first_gap_idx = gap_indices[0]
        second_gap_idx = gap_indices[1]
        print(f"  Test: Keeping second segment between gaps (rows {first_gap_idx} to {second_gap_idx}, {second_gap_idx - first_gap_idx} rows).")
        processed_df = processed_df.iloc[first_gap_idx:second_gap_idx].reset_index(drop=True)
    elif len(gap_indices) == 1:
        first_gap_idx = gap_indices[0]
        print(f"  Test: Only one gap found. Keeping data after first gap (rows {first_gap_idx} onwards, {len(processed_df) - first_gap_idx} rows).")
        processed_df = processed_df.iloc[first_gap_idx:].reset_index(drop=True)
    else:
        print(f"  Test: No gaps found. Using all data ({len(processed_df)} rows).")
    
    return processed_df

In [4]:
# Load collected bike data for training
print("Loading TRAIN data:")
collected_bike_train = load_collected_data_train('data/collected/bike')

# Load all collected car data for training
collected_car_train_dfs = []
for i in range(1, 2):
    car_folder = f'data/collected/car-{i}'
    car_df = load_collected_data_train(car_folder)
    collected_car_train_dfs.append(car_df)

collected_car_train = pd.concat(collected_car_train_dfs, ignore_index=True)

print(f"\nLoaded collected bike train: {len(collected_bike_train)} rows")
print(f"Total collected car train data: {len(collected_car_train)} rows")

# Load collected bike data for testing
print("\n" + "="*60)
print("Loading TEST data:")
collected_bike_test = load_collected_data_test('data/collected/bike')

# Load all collected car data for testing
collected_car_test_dfs = []
for i in range(1, 2):
    car_folder = f'data/collected/car-{i}'
    car_df = load_collected_data_test(car_folder)
    collected_car_test_dfs.append(car_df)

collected_car_test = pd.concat(collected_car_test_dfs, ignore_index=True)

print(f"\nLoaded collected bike test: {len(collected_bike_test)} rows")
print(f"Total collected car test data: {len(collected_car_test)} rows")

Loading TRAIN data:
  Train: Time gap detected at index 1631. Keeping first segment (1631 rows).
  Train: Time gap detected at index 77240. Keeping first segment (77240 rows).

Loaded collected bike train: 1631 rows
Total collected car train data: 77240 rows

Loading TEST data:
  Test: Keeping second segment between gaps (rows 1631 to 2498, 867 rows).
  Test: Keeping second segment between gaps (rows 77240 to 85954, 8714 rows).

Loaded collected bike test: 867 rows
Total collected car test data: 8714 rows


In [5]:
# Process and save collected train dataset
collected_train_dataset = process_vehicle_dataset(
    [collected_bike_train, collected_car_train],
    ['bike', 'car'],
    'data/vehicle_data_collected_train.pkl',
)

Processed bike: 15 windows with label value 0
Processed car: 773 windows with label value 1

Dataset saved to data/vehicle_data_collected_train.pkl
Total samples: 788
Data shape: torch.Size([788, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [6]:
# Process and save collected test dataset
collected_test_dataset = process_vehicle_dataset(
    [collected_bike_test, collected_car_test],
    ['bike', 'car'],
    'data/vehicle_data_collected_test.pkl',
)

Processed bike: 7 windows with label value 0
Processed car: 85 windows with label value 1

Dataset saved to data/vehicle_data_collected_test.pkl
Total samples: 92
Data shape: torch.Size([92, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [7]:
# Compare collected dataset statistics
print("\n" + "="*60)
print("COLLECTED DATASET COMPARISON")
print("="*60)

collected_train_total = collected_train_dataset['data'].shape[0]
collected_test_total = collected_test_dataset['data'].shape[0]

print(f"\nTotal windows in collected train set: {collected_train_total}")
print(f"Total windows in collected test set: {collected_test_total}")
print(f"Ratio (test/train): {collected_test_total/collected_train_total:.2%}")

print(f"\nCollected train label distribution: {torch.bincount(collected_train_dataset['label'])}")
print(f"Collected test label distribution: {torch.bincount(collected_test_dataset['label'])}")


COLLECTED DATASET COMPARISON

Total windows in collected train set: 788
Total windows in collected test set: 92
Ratio (test/train): 11.68%

Collected train label distribution: tensor([ 15, 773])
Collected test label distribution: tensor([ 7, 85])


## Load Kaggle Dataset

In [8]:
# Load raw data
bike_df = pd.read_csv('data/kaggle/bike.csv')

# Load all car data files
car_clear_df = pd.read_csv('data/kaggle/car-clear.csv')
# car_clear2_df = pd.read_csv('data/kaggle/car-clear2.csv')
# car_rain_df = pd.read_csv('data/kaggle/car-rain.csv')
# car_city_df = pd.read_csv('data/kaggle/car-city-newark-light-rain.csv')

# Concatenate all car data
car_df = car_clear_df

print(f"Loaded bike: {len(bike_df)} rows")
print(f"Loaded car-clear: {len(car_clear_df)} rows")
print(f"Total car data: {len(car_df)} rows")

Loaded bike: 35515 rows
Loaded car-clear: 44416 rows
Total car data: 44416 rows


In [9]:
split_ratio = 0.8

# Split by time instead of by row index
bike_split_time = bike_df['time'].min() + (bike_df['time'].max() - bike_df['time'].min()) * split_ratio
bike_train = bike_df[bike_df['time'] <= bike_split_time].reset_index(drop=True)
bike_test = bike_df[bike_df['time'] > bike_split_time].reset_index(drop=True)

# Car split
car_split_time = car_df['time'].min() + (car_df['time'].max() - car_df['time'].min()) * split_ratio
car_train = car_df[car_df['time'] <= car_split_time].reset_index(drop=True)
car_test = car_df[car_df['time'] > car_split_time].reset_index(drop=True)

print("Train set:")
print(f"  Bike: {len(bike_train)} rows")
print(f"  Car: {len(car_train)} rows")

print("\nTest set:")
print(f"  Bike: {len(bike_test)} rows")
print(f"  Car: {len(car_test)} rows")

Train set:
  Bike: 28411 rows
  Car: 35854 rows

Test set:
  Bike: 7104 rows
  Car: 8562 rows


In [10]:
train_dataset = process_vehicle_dataset(
    [bike_train, car_train],
    ['bike', 'car'],
    'data/vehicle_data_kaggle_train.pkl',
)

Processed bike: 239 windows with label value 0
Processed car: 319 windows with label value 1

Dataset saved to data/vehicle_data_kaggle_train.pkl
Total samples: 558
Data shape: torch.Size([558, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [11]:
test_dataset = process_vehicle_dataset(
    [bike_test, car_test],
    ['bike', 'car'],
    'data/vehicle_data_kaggle_test.pkl',
)

Processed bike: 59 windows with label value 0
Processed car: 79 windows with label value 1

Dataset saved to data/vehicle_data_kaggle_test.pkl
Total samples: 138
Data shape: torch.Size([138, 3, 50])
Label mapping: {'bike': 0, 'car': 1}


In [12]:
# Compare final window counts
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)

train_total = train_dataset['data'].shape[0]
test_total = test_dataset['data'].shape[0]

print(f"\nTotal windows in train set: {train_total}")
print(f"Total windows in test set: {test_total}")
print(f"Ratio (test/train): {test_total/train_total:.2%}")

print(f"\nTrain label distribution: {torch.bincount(train_dataset['label'])}")
print(f"Test label distribution: {torch.bincount(test_dataset['label'])}")


FINAL COMPARISON

Total windows in train set: 558
Total windows in test set: 138
Ratio (test/train): 24.73%

Train label distribution: tensor([239, 319])
Test label distribution: tensor([59, 79])
